# ADNI Advanced Survival Pipeline — Run All

**One-shot notebook.** Pressing *Restart & Run All* takes a fresh
clone of the repo from raw DVFs + master CSV to a trained model,
evaluated checkpoint, and comparison table.

| Phase | What happens |
|------:|-------------|
| 0 | Environment + MPS env vars (set before `import torch`) |
| 1 | Config, device, seed |
| 2 | DVF folder discovery + CSV-driven labels + ID-aware save |
| 3 | Stratified train/val split (saved with subject IDs) |
| 4 | DVF normalization stats from training split |
| 5 | Datasets and DataLoaders (Apple-Silicon-friendly) |
| 6 | IPCW + CHI |
| 7 | Model + Trainer |
| 8 | Stage 2 fine-tune |
| 9 | Evaluation + comparison table + survival/CIF curves |

**Apple Silicon notes** — this notebook is tuned for `torch==2.2.2`
on MPS (the version pinned by the project's dependency chain
`torch==2.2.2 → numpy<2.0 → scikit-survival==0.22.2`). On that
version MPS autocast and GradScaler are unavailable, so training
runs in **float32**. The patched trainer auto-detects a newer
PyTorch and will switch to fp16 autocast on MPS when available.

## Phase 0 — Environment

MPS-specific environment variables **must** be set before `torch` is
imported, otherwise they have no effect.

In [1]:
import os
# Fall back to CPU for ops that don't yet have MPS kernels
# (ConvTranspose3D is the notable one for 3D work).
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
# Let PyTorch use as much unified memory as macOS will give it.
os.environ.setdefault("PYTORCH_MPS_HIGH_WATERMARK_RATIO", "0.0")

import sys, random, logging, platform
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# Repo root — works whether CWD is Transformer/ or the repo root.
_cwd = Path().resolve()
REPO_ROOT = _cwd.parent if _cwd.name == "Transformer" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(str(REPO_ROOT))

# macOS multiprocessing — must be 'spawn' for workers + MPS.
import torch.multiprocessing as mp
try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass  # already set

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("run_all")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print(f"Python    : {sys.version.split()[0]}")
print(f"torch     : {torch.__version__}")
print(f"System    : {platform.platform()}")
print(f"CUDA      : {torch.cuda.is_available()}")
print(f"MPS       : {torch.backends.mps.is_available()}")
print(f"Repo root : {REPO_ROOT}")

Python    : 3.11.11
torch     : 2.2.2
System    : macOS-26.4.1-arm64-arm-64bit
CUDA      : False
MPS       : True
Repo root : /Users/nchavez/Projects/school/Classes/COMP549MDSProject/adni-survival-analysis-main


### Cleanup previous runs

Remove stale checkpoints, labels, and caches from prior training
runs so they cannot influence this run. Safe to skip if this is
your first run.

In [2]:
import shutil

CLEAN_DIRS = [
    REPO_ROOT / "Transformer" / "checkpoints",
    REPO_ROOT / "Transformer" / "split_labels",
    REPO_ROOT / "Transformer" / "val_labels",
    REPO_ROOT / "Transformer" / "derived_labels",
    REPO_ROOT / "Transformer" / "outputs",
    REPO_ROOT / "Transformer" / "figures",
]

for d in CLEAN_DIRS:
    if d.exists():
        n = sum(1 for _ in d.iterdir())
        if n > 0:
            shutil.rmtree(d)
            d.mkdir(parents=True)
            print(f"  Cleared {d.relative_to(REPO_ROOT)} ({n} items)")
        else:
            print(f"  {d.relative_to(REPO_ROOT)}: already empty")
    else:
        d.mkdir(parents=True, exist_ok=True)
        print(f"  {d.relative_to(REPO_ROOT)}: created")

print("\nAll output directories clean. Ready for a fresh run.")

  Transformer/checkpoints: already empty
  Cleared Transformer/split_labels (3 items)
  Cleared Transformer/val_labels (3 items)
  Transformer/derived_labels: already empty
  Transformer/outputs: already empty
  Transformer/figures: already empty

All output directories clean. Ready for a fresh run.


## Phase 1 — Configuration

A single `ModelConfig` dataclass consolidates every hyperparameter.
Device selection prefers MPS on Apple Silicon, CUDA on Linux with a
GPU, CPU elsewhere.

In [3]:
from Transformer.config.model_config import ModelConfig, DEVICE

config = ModelConfig(
    dvf_dir              = REPO_ROOT / "data" / "dvf_structured",
    survival_labels_dir  = REPO_ROOT / "Transformer" / "split_labels",
    checkpoint_dir       = REPO_ROOT / "Transformer" / "checkpoints",
    figures_dir          = REPO_ROOT / "Transformer" / "figures",
    # Training scope — adjust upward for a final run.
    v_max                    = 5,
    d_model                  = 256,
    max_epochs_stage2        = 20,
    early_stopping_patience  = 5,
)
config.validate()
assert config.device == DEVICE, "Config/DEVICE mismatch"
print(f"Device = {config.device}")

Device = mps


## Phase 2 — DVF folder discovery + CSV-driven labels

We read the master CSV, derive MCI→Dementia survival labels for
every subject that ALSO has a DVF folder, and save the three-part
label artifact `(mci_y_duration.npy, mci_y_event.npy,
mci_subject_ids.npy)` to disk. The ID file is the join key that
kills the positional-matching bug.

**If your DVFs live flat in `data/adni-flow-tensors/`**, the first
cell below restructures them into per-subject folders via symlinks
(no data duplication). Skip if that's already done.

In [4]:
import re

# Optional: restructure flat DVFs into per-subject folders
FLOW_DIR   = REPO_ROOT / "data" / "adni-flow-tensors"
STRUCT_DIR = config.dvf_dir
VISIT_MONTHS = {
    "bl":0, "m03":3, "m06":6, "m12":12, "m18":18,
    "m24":24, "m36":36, "m48":48, "m60":60,
}

if STRUCT_DIR.exists() and any(STRUCT_DIR.iterdir()):
    n = sum(1 for d in STRUCT_DIR.iterdir() if d.is_dir())
    print(f"Structured dir already exists — {n} subject folders")
elif FLOW_DIR.exists():
    STRUCT_DIR.mkdir(parents=True, exist_ok=True)
    for f in sorted(FLOW_DIR.glob("*_flow.npy")):
        m = re.match(r"(\d+_S_\d+)_(.+)_flow", f.stem)
        if not m: continue
        month = VISIT_MONTHS.get(m.group(2))
        if month is None: continue
        subj_dir = STRUCT_DIR / m.group(1)
        subj_dir.mkdir(exist_ok=True)
        dst = subj_dir / f"{month:03d}.npy"
        if not dst.exists():
            os.symlink(str(f.resolve()), str(dst))
    n = sum(1 for d in STRUCT_DIR.iterdir() if d.is_dir())
    print(f"Created {n} subject folders via symlinks")
else:
    raise FileNotFoundError(
        f"Neither {STRUCT_DIR} nor {FLOW_DIR} exists. "
        f"Place your DVFs in one of these."
    )

Structured dir already exists — 1235 subject folders


In [5]:
# Build (PTID, duration, event) from the master CSV, keeping only
# MCI-at-baseline subjects that ALSO have a DVF folder.
MASTER_CSV = REPO_ROOT / "data" / "master_data_improved_04052026_v3.csv"
df = pd.read_csv(str(MASTER_CSV), low_memory=False)
print(f"Master CSV: {len(df):,} rows, {df['PTID'].nunique()} patients")

df_bl  = df[df["VISCODE"] == "bl"].drop_duplicates("PTID").copy()
mci_bl = df_bl[df_bl["DX_bl"].isin(["LMCI", "EMCI"])].copy()
print(f"MCI-at-baseline: {len(mci_bl)}")

dvf_subjects = {d.name for d in config.dvf_dir.iterdir() if d.is_dir()}
print(f"DVF folders    : {len(dvf_subjects)}")

records = []
for _, bl in mci_bl.iterrows():
    ptid = bl["PTID"]
    if ptid not in dvf_subjects: continue
    subj = df[df["PTID"] == ptid].sort_values("Month")
    dem  = subj[(subj["VISCODE"] != "bl") & (subj["DX"] == "Dementia")]
    t, e = (float(dem["Month"].min()), 1) if len(dem) else (float(subj["Month"].max()), 0)
    if t > 0:
        records.append({"PTID": str(ptid), "duration_months": t, "event": e})

surv = pd.DataFrame(records)
print(f"\nSubjects with MCI label AND DVF folder: {len(surv)}")
print(f"  Events (→ Dementia): {(surv['event']==1).sum()}")
print(f"  Censored            : {(surv['event']==0).sum()}")

Master CSV: 16,421 rows, 2430 patients
MCI-at-baseline: 1113
DVF folders    : 1235

Subjects with MCI label AND DVF folder: 522
  Events (→ Dementia): 230
  Censored            : 292


## Phase 3 — Stratified split + save with subject IDs

The split is saved through `save_survival_labels(...)` which writes
all three files atomically — labels can never get out of sync with
their IDs again.

In [6]:
from sklearn.model_selection import train_test_split
from Transformer.data.label_io import save_survival_labels

ptids    = surv["PTID"].tolist()
dur_all  = surv["duration_months"].values.astype(np.float32)
evt_all  = surv["event"].values.astype(np.int64)

idx = np.arange(len(ptids))
train_idx, val_idx = train_test_split(
    idx, test_size=0.2, random_state=SEED, stratify=evt_all,
)

train_sids = [ptids[i] for i in train_idx]
val_sids   = [ptids[i] for i in val_idx]
train_dur, val_dur = dur_all[train_idx], dur_all[val_idx]
train_evt, val_evt = evt_all[train_idx], evt_all[val_idx]

train_dir = REPO_ROOT / "Transformer" / "split_labels"
val_dir   = REPO_ROOT / "Transformer" / "val_labels"

save_survival_labels(train_dir, train_sids, train_dur, train_evt)
save_survival_labels(val_dir,   val_sids,   val_dur,   val_evt)

print(f"Train: {len(train_sids)} (events {(train_evt==1).sum()})  → {train_dir}")
print(f"Val  : {len(val_sids)} (events {(val_evt==1).sum()})  → {val_dir}")

11:59:40 | INFO | Saved 417 labeled subjects to /Users/nchavez/Projects/school/Classes/COMP549MDSProject/adni-survival-analysis-main/Transformer/split_labels (with subject ID index)
11:59:40 | INFO | Saved 105 labeled subjects to /Users/nchavez/Projects/school/Classes/COMP549MDSProject/adni-survival-analysis-main/Transformer/val_labels (with subject ID index)


Train: 417 (events 184)  → /Users/nchavez/Projects/school/Classes/COMP549MDSProject/adni-survival-analysis-main/Transformer/split_labels
Val  : 105 (events 46)  → /Users/nchavez/Projects/school/Classes/COMP549MDSProject/adni-survival-analysis-main/Transformer/val_labels


## Phase 4 — DVF normalization stats (training split only)

In [7]:
from Transformer.data.dvf_dataset import NormalizationStats

train_paths = []
for sid in train_sids:
    train_paths.extend(sorted((config.dvf_dir / sid).glob("*.npy")))
print(f"Computing norm stats from {len(train_paths)} training DVFs …")
norm_stats = NormalizationStats.compute(train_paths)
for c, ax in enumerate(["Δx", "Δy", "Δz"]):
    print(f"  {ax}: μ={norm_stats.mean[c]:+.4f}  σ={norm_stats.std[c]:.4f}")

Computing norm stats from 1739 training DVFs …
  Δx: μ=+0.0017  σ=0.5677
  Δy: μ=+0.0057  σ=0.7362
  Δz: μ=+0.0020  σ=1.4940


## Phase 5 — Datasets and DataLoaders

Apple-Silicon-friendly settings:
- `multiprocessing_context="spawn"` on macOS (fork + Metal = deadlock).
- `persistent_workers=True` avoids per-epoch spawn overhead.
- `pin_memory` is only meaningful on CUDA — we guard it.

In [8]:
from torch.utils.data import DataLoader
from Transformer.data.dvf_dataset import LongitudinalDVFDataset

train_ds = LongitudinalDVFDataset(
    subject_ids=train_sids,
    dvf_dir=config.dvf_dir, config=config,
    norm_stats=norm_stats,
    survival_labels_dir=train_dir,
)
val_ds = LongitudinalDVFDataset(
    subject_ids=val_sids,
    dvf_dir=config.dvf_dir, config=config,
    norm_stats=norm_stats,
    survival_labels_dir=val_dir,
)
print(f"Train dataset: {len(train_ds)} subjects")
print(f"Val dataset  : {len(val_ds)} subjects")

is_mac = (platform.system() == "Darwin")
is_cuda = (config.device.type == "cuda")
NW = 2  # 0 on notebooks is a perf bug — spawn is safe.
mp_ctx = "spawn" if (is_mac and NW > 0) else None

train_loader = DataLoader(
    train_ds,
    batch_size=config.batch_size_physical, shuffle=True,
    num_workers=NW, persistent_workers=(NW > 0),
    multiprocessing_context=mp_ctx, pin_memory=is_cuda,
    collate_fn=LongitudinalDVFDataset.collate_fn,
)
val_loader = DataLoader(
    val_ds,
    batch_size=config.batch_size_physical, shuffle=False,
    num_workers=NW, persistent_workers=(NW > 0),
    multiprocessing_context=mp_ctx, pin_memory=is_cuda,
    collate_fn=LongitudinalDVFDataset.collate_fn,
)
print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

# Spot-check — one sample, to confirm shapes.
s = train_ds[0]
print(f"\nSample 0: subject={s['subject_id']}  dvf={tuple(s['dvf_sequence'].shape)}  "
      f"dur={s['duration'].item():.0f}mo  evt={s['event'].item()}")

12:01:14 | INFO | LongitudinalDVFDataset ready: 417 subjects, 1739 total visits, v_max=5
12:01:14 | INFO | LongitudinalDVFDataset ready: 105 subjects, 428 total visits, v_max=5


Train dataset: 417 subjects
Val dataset  : 105 subjects
Train batches: 209  |  Val batches: 53

Sample 0: subject=099_S_0958  dvf=(5, 3, 128, 128, 128)  dur=6mo  evt=0


## Phase 6 — IPCW estimator + CHI interpolator

Fit the censoring-survival KM on training **subjects** only.
Feeding `train_ds.durations` / `train_ds.events` directly avoids
the old bug of iterating the whole dataset (loading every DVF
volume from disk) just to pull scalars.

In [9]:
from Transformer.losses.ipcw_loss import CensoringSurvivalEstimator
from Transformer.utils.chi_interpolation import ConstantHazardInterpolator

censoring_est = CensoringSurvivalEstimator()
censoring_est.fit(train_ds.durations, train_ds.events)
chi = ConstantHazardInterpolator.from_config(config)

for t in config.t_grid:
    g = float(censoring_est(np.array([t], dtype=np.float64))[0])
    print(f"  G({t:3d} mo) = {g:.4f}")

12:01:15 | INFO | CensoringSurvivalEstimator fitted on 417 subjects (55.9% censored).


  G( 12 mo) = 0.8962
  G( 24 mo) = 0.8166
  G( 36 mo) = 0.6175
  G( 48 mo) = 0.4900
  G( 60 mo) = 0.3960


## Phase 7 — Model + trainer

The `ADNISurvivalPipeline` is the full BrainIAC → sequence (Longformer
by default) → PMA → TraCeR stack. We default to the single-Longformer
variant here; pass `sequence_model_cls=None` to `ADNISurvivalPipeline`
to get the parallel gated fusion of Longformer + Mamba.

**n_risks note.** The label-builder emits events `{0, 1}`
(censored / dementia), so `n_risks=1` (the default) is correct.
If you later add death-without-dementia subjects with event code `2`,
bump `n_risks=2` in ModelConfig and the architecture seamlessly grows
a second cause head.

In [10]:
from Transformer.models.longformer_sequence import LongformerSequence
from Transformer.models.pipeline import ADNISurvivalPipeline
from Transformer.training.trainer import LPFTTrainer

pipeline = ADNISurvivalPipeline(
    config=config,
    sequence_model_cls=LongformerSequence,
    pretrained_brainiac_path=None,  # MedicalNet auto-downloads
).to(config.device)

n_total    = sum(p.numel() for p in pipeline.parameters())
n_trainable = sum(p.numel() for p in pipeline.parameters() if p.requires_grad)
print(f"Params — total: {n_total:,} | trainable: {n_trainable:,} | frozen: {n_total - n_trainable:,}")

trainer = LPFTTrainer(
    model=pipeline, config=config,
    train_loader=train_loader, val_loader=val_loader,
    censoring_estimator=censoring_est, chi_interpolator=chi,
    train_durations=train_ds.durations,
    train_events=train_ds.events,
    norm_stats=norm_stats,
    device=str(config.device),
)
print("Trainer ready")

12:01:15 | WARNING | mamba_ssm package not found — using sequential PyTorch fallback for selective scan. For production use on CUDA, install via: pip install mamba-ssm causal-conv1d
12:01:17 | INFO | Loading MedicalNet pretrained model from https://huggingface.co/TencentMedicalNet/MedicalNet-Resnet50
12:01:18 | INFO | resnet_50_23dataset.pth downloaded
12:01:18 | INFO | Loaded MedicalNet pretrained ResNet50 (23 datasets, including brain MRI). This backbone provides strong 3D medical imaging features without BrainIAC weights.
12:01:18 | INFO | Adapted conv1: [64, 1, 7, 7, 7] → [64, 3, 7, 7, 7] via W/3 duplication.
12:01:18 | INFO | Replaced 53 BatchNorm3d layers with GroupNorm(num_groups=16)
12:01:18 | INFO | Gradient checkpointing enabled for layer3 + layer4
12:01:18 | INFO | Built native SDPA LongformerSequence (heads=8, layers=6, window=512). No HuggingFace dependency.
12:01:18 | INFO | Single sequence model: LongformerSequence
12:01:18 | INFO | ADNISurvivalPipeline assembled: extrac

Params — total: 54,396,869 | trainable: 8,197,893 | frozen: 46,198,976
Trainer ready


## Phase 8 — Fine-tune (Stage 2)

Stage 1 (linear probe with frozen sequence model) is skipped
intentionally — the BrainIAC backbone is pretrained and a frozen
Longformer/Mamba provides no useful temporal signal at init. Start
with sequence model + PMA + head all trainable.

In [11]:
print("=" * 60)
print("STAGE 2 — Full Fine-Tune")
print("=" * 60)
best = trainer.run_stage2()
print(f"\nBest C_td = {best.get('c_td', 0):.4f} @ epoch {trainer.best_epoch}")

12:01:18 | INFO | ═══ Stage 2: Fine-Tune (up to 20 epochs) ═══
12:01:18 | INFO | MPS (torch 2.2.2 < 2.5): BrainIAC extractor moved to CPU — Conv3D autograd not supported on MPS. Temporal model + survival head remain on MPS.
12:01:18 | INFO | Stage 2 param groups: 8197893 trainable parameters


STAGE 2 — Full Fine-Tune


13:07:23 | INFO | Metrics — C_td=0.4611, Uno_C=0.4701 (tau=36.0), IBS=0.3560
13:07:24 | INFO | Checkpoint: /Users/nchavez/Projects/school/Classes/COMP549MDSProject/adni-survival-analysis-main/Transformer/checkpoints/checkpoint_best.pt (epoch=1, C_td=0.4611)
13:07:24 | INFO | Stage2 Epoch 1/20 — loss=1.7764, C_td=0.4611 ★ (new best), Uno_C=0.4701, IBS=0.3560
14:12:55 | INFO | Metrics — C_td=0.4771, Uno_C=0.4795 (tau=36.0), IBS=0.2532
14:12:55 | INFO | Checkpoint: /Users/nchavez/Projects/school/Classes/COMP549MDSProject/adni-survival-analysis-main/Transformer/checkpoints/checkpoint_best.pt (epoch=2, C_td=0.4771)
14:12:55 | INFO | Stage2 Epoch 2/20 — loss=1.1506, C_td=0.4771 ★ (new best), Uno_C=0.4795, IBS=0.2532


KeyboardInterrupt: 

## Phase 9 — Evaluation + comparison table + curves

In [ ]:
from Transformer.metrics.concordance import compute_all_metrics

pipeline.eval()
haz_list = []
with torch.no_grad():
    for batch in val_loader:
        if config.device.type == "mps":
            torch.mps.empty_cache()
        out = pipeline(
            batch["dvf_sequence"].to(config.device),
            batch["time_deltas"].to(config.device),
            batch["missing_mask"].to(config.device),
        )
        haz_list.append(out["hazards"].float().cpu())
val_hazards = torch.cat(haz_list, dim=0)

final = compute_all_metrics(
    val_hazards, val_ds.durations, val_ds.events,
    train_ds.durations, train_ds.events,
    chi, pipeline.survival_head, config,
)

print("─" * 45)
print(f"  Antolini C_td : {final['c_td']:.4f}")
print(f"  Uno C_τ       : {final['uno_c']:.4f}  (τ={final['tau']:.0f} mo)")
print(f"  IBS           : {final['ibs']:.4f}")
print("─" * 45)

In [ ]:
from Transformer.metrics.concordance import format_comparison_table
from IPython.display import Markdown, display
import csv

md_table, csv_rows = format_comparison_table(final, cohort="MCI→Dementia")
display(Markdown(md_table))

out_dir = REPO_ROOT / "Transformer" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "model_comparison_transformer.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=csv_rows[0].keys())
    w.writeheader(); w.writerows(csv_rows)
print(f"Saved → {csv_path}")

In [ ]:
import matplotlib.pyplot as plt

config.figures_dir.mkdir(parents=True, exist_ok=True)
t_grid = np.array(config.t_grid)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
rng = np.random.default_rng(SEED)
pick = rng.choice(len(val_hazards), size=min(3, len(val_hazards)), replace=False)

for idx in pick:
    h = val_hazards[idx:idx+1]
    with torch.no_grad():
        S   = pipeline.survival_head.hazards_to_survival(h)[0].numpy()
        cif = pipeline.survival_head.hazards_to_cif(h)[0, 0].numpy()
    sid = val_ds.subject_ids[idx]
    axes[0].plot(t_grid, S,   marker="o", label=f"{sid}")
    axes[1].plot(t_grid, cif, marker="s", label=f"{sid}")

for ax, title, ylabel in [
    (axes[0], "Overall Survival S(t)", "Probability"),
    (axes[1], "CIF — MCI→Dementia",    "Cumulative Incidence"),
]:
    ax.set_xlabel("Months"); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.set_ylim(-0.02, 1.05); ax.grid(alpha=0.3); ax.legend(fontsize=8)

plt.tight_layout()
fig_path = config.figures_dir / "survival_cif_curves.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {fig_path}")

## Done

| Artifact | Location |
|----------|----------|
| Best checkpoint | `Transformer/checkpoints/checkpoint_best.pt` |
| Final checkpoint | `Transformer/checkpoints/checkpoint_final.pt` |
| Comparison CSV | `Transformer/outputs/model_comparison_transformer.csv` |
| Survival + CIF figures | `Transformer/figures/survival_cif_curves.png` |
| Train labels + IDs | `Transformer/split_labels/{duration,event,subject_ids}.npy` |
| Val labels + IDs | `Transformer/val_labels/{duration,event,subject_ids}.npy` |

**To resume training:** `trainer.load_checkpoint(...)` then
`trainer.run_stage2()`.

**To swap to Mamba:** change the `sequence_model_cls` argument in
Phase 7 from `LongformerSequence` to `Mamba3DSequence`, or pass
`sequence_model_cls=None` for parallel gated fusion of both.